# DATATHON 2026 Round 1
## Part 2 - Visualization and Data Analysis

**Team:** `<CT-PTIT/> CapyData`

Phần phân tích chọn 6 góc nhìn chính của một doanh nghiệp thời trang thương mại điện tử:

1. Doanh thu và gross margin thay đổi ra sao theo thời gian?
2. Nhóm sản phẩm nào đóng góp nhiều nhất vào tăng trưởng có lợi nhuận?
3. Khuyến mãi đang hỗ trợ tăng trưởng hay làm mỏng margin?
4. Rủi ro sau bán hàng đến từ trả hàng, giao vận hay trải nghiệm khách hàng?
5. Nguồn traffic nào có chất lượng tốt hơn?
6. Nhóm hàng nào có nguy cơ thiếu hàng hoặc tồn kho cao?


In [ ]:
# Required packages: pandas, numpy, matplotlib, seaborn
# Uncomment the next line only if the environment does not have these packages
# %pip install pandas numpy matplotlib seaborn

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 11

DATA_DIR = Path('.')
FIG_DIR = Path('eda_figures')
FIG_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

## 1. Data Loading

Đọc các bảng dữ liệu master, transaction, analytical và operational. Các bảng này được dùng để dựng chỉ số bán hàng, sản phẩm, khuyến mãi, vận hành và traffic.


In [ ]:
products = pd.read_csv(DATA_DIR / 'products.csv')
customers = pd.read_csv(DATA_DIR / 'customers.csv', parse_dates=['signup_date'])
promotions = pd.read_csv(DATA_DIR / 'promotions.csv', parse_dates=['start_date', 'end_date'])
geography = pd.read_csv(DATA_DIR / 'geography.csv')

orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')
payments = pd.read_csv(DATA_DIR / 'payments.csv')
shipments = pd.read_csv(DATA_DIR / 'shipments.csv', parse_dates=['ship_date', 'delivery_date'])
returns = pd.read_csv(DATA_DIR / 'returns.csv', parse_dates=['return_date'])
reviews = pd.read_csv(DATA_DIR / 'reviews.csv', parse_dates=['review_date'])

sales = pd.read_csv(DATA_DIR / 'sales.csv', parse_dates=['Date'])
inventory = pd.read_csv(DATA_DIR / 'inventory.csv', parse_dates=['snapshot_date'])
web = pd.read_csv(DATA_DIR / 'web_traffic.csv', parse_dates=['date'])

summary = pd.DataFrame({
    'table': ['products','customers','promotions','geography','orders','order_items','payments','shipments','returns','reviews','sales','inventory','web_traffic'],
    'rows': [len(products), len(customers), len(promotions), len(geography), len(orders), len(order_items), len(payments), len(shipments), len(returns), len(reviews), len(sales), len(inventory), len(web)],
    'columns': [products.shape[1], customers.shape[1], promotions.shape[1], geography.shape[1], orders.shape[1], order_items.shape[1], payments.shape[1], shipments.shape[1], returns.shape[1], reviews.shape[1], sales.shape[1], inventory.shape[1], web.shape[1]]
})
display(summary)

## 2. Item-Level Analytical Table

Tạo bảng `df` ở cấp sản phẩm trong từng đơn hàng. Bảng này nối order item với sản phẩm, đơn hàng, địa lý, thanh toán, giao vận, trả hàng và review; sau đó tính net revenue, gross profit, margin, discount rate, delivery time và return rate.


In [ ]:
def safe_divide(num, den):
    return np.where(den == 0, np.nan, num / den)

orders_geo = orders.merge(geography, on='zip', how='left')

return_agg = returns.groupby(['order_id', 'product_id'], as_index=False).agg(
    returned_qty=('return_quantity', 'sum'),
    refund_amount=('refund_amount', 'sum'),
    return_records=('return_id', 'count')
)

review_agg = reviews.groupby(['order_id', 'product_id'], as_index=False).agg(
    avg_rating=('rating', 'mean'),
    review_count=('review_id', 'count')
)

df = (
    order_items
    .merge(products, on='product_id', how='left')
    .merge(orders_geo, on='order_id', how='left')
    .merge(payments[['order_id', 'payment_value', 'installments']], on='order_id', how='left')
    .merge(shipments, on='order_id', how='left')
    .merge(return_agg, on=['order_id', 'product_id'], how='left')
    .merge(review_agg, on=['order_id', 'product_id'], how='left')
)

for col in ['returned_qty', 'refund_amount', 'return_records', 'review_count']:
    df[col] = df[col].fillna(0)

df['gross_revenue'] = df['quantity'] * df['unit_price']
df['net_revenue'] = df['gross_revenue'] - df['discount_amount']
df['line_cogs'] = df['quantity'] * df['cogs']
df['gross_profit'] = df['net_revenue'] - df['line_cogs']
df['gross_margin_rate'] = safe_divide(df['gross_profit'], df['net_revenue'])
df['discount_rate'] = safe_divide(df['discount_amount'], df['gross_revenue'])
df['return_flag'] = (df['returned_qty'] > 0).astype(int)
df['return_rate_qty'] = safe_divide(df['returned_qty'], df['quantity'])
df['has_promo'] = df['promo_id'].notna() & (df['promo_id'].astype(str).str.len() > 0)
df['has_second_promo'] = df['promo_id_2'].notna() & (df['promo_id_2'].astype(str).str.len() > 0)
df['delivery_days'] = (df['delivery_date'] - df['ship_date']).dt.days

df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['year_month'] = df['order_date'].dt.to_period('M').astype(str)
df['weekday'] = df['order_date'].dt.day_name()

print(df.shape)
display(df.head())

## Insight 1 - Revenue and Margin Dynamics

Câu hỏi: doanh nghiệp tăng trưởng như thế nào, có mùa vụ không, và tăng trưởng có đi kèm lợi nhuận không?


In [ ]:
sales2 = sales.copy()
sales2['gross_profit'] = sales2['Revenue'] - sales2['COGS']
sales2['margin_rate'] = safe_divide(sales2['gross_profit'], sales2['Revenue'])
sales2['year_month'] = sales2['Date'].dt.to_period('M').dt.to_timestamp()
sales2['year'] = sales2['Date'].dt.year
sales2['month'] = sales2['Date'].dt.month

monthly_sales = sales2.groupby('year_month', as_index=False).agg(
    revenue=('Revenue', 'sum'),
    cogs=('COGS', 'sum'),
    gross_profit=('gross_profit', 'sum')
)
monthly_sales['margin_rate'] = safe_divide(monthly_sales['gross_profit'], monthly_sales['revenue'])
monthly_sales['revenue_ma3'] = monthly_sales['revenue'].rolling(3, min_periods=1).mean()
monthly_sales['profit_ma3'] = monthly_sales['gross_profit'].rolling(3, min_periods=1).mean()

yearly_sales = sales2.groupby('year', as_index=False).agg(
    revenue=('Revenue', 'sum'),
    gross_profit=('gross_profit', 'sum')
)
yearly_sales['margin_rate'] = safe_divide(yearly_sales['gross_profit'], yearly_sales['revenue'])

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(monthly_sales['year_month'], monthly_sales['revenue_ma3'], label='Revenue - 3M moving average', linewidth=2.2)
ax1.plot(monthly_sales['year_month'], monthly_sales['profit_ma3'], label='Gross profit - 3M moving average', linewidth=2.2)
ax1.set_title('Monthly Revenue and Gross Profit Trend')
ax1.set_xlabel('Month')
ax1.set_ylabel('Value')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(monthly_sales['year_month'], monthly_sales['margin_rate'], color='black', alpha=0.35, label='Margin rate')
ax2.set_ylabel('Gross margin rate')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIG_DIR / '01_revenue_margin_trend.png', dpi=180, bbox_inches='tight')
plt.show()

peak_month = monthly_sales.loc[monthly_sales['revenue'].idxmax()]
low_margin_month = monthly_sales.loc[monthly_sales['margin_rate'].idxmin()]
best_year = yearly_sales.loc[yearly_sales['revenue'].idxmax()]
first_year = yearly_sales.iloc[0]
last_year = yearly_sales.iloc[-1]
revenue_growth = (last_year['revenue'] / first_year['revenue'] - 1) * 100

msg = f'''
**Điểm chính**

- Tổng doanh thu năm đầu ({int(first_year['year'])}) là **{first_year['revenue']:,.0f}**, năm cuối ({int(last_year['year'])}) là **{last_year['revenue']:,.0f}**, tương đương thay đổi **{revenue_growth:,.1f}%**.
- Tháng có doanh thu cao nhất là **{peak_month['year_month'].strftime('%Y-%m')}** với doanh thu **{peak_month['revenue']:,.0f}**.
- Tháng có margin thấp nhất là **{low_margin_month['year_month'].strftime('%Y-%m')}**, margin chỉ **{low_margin_month['margin_rate']:.2%}**.
- Năm doanh thu lớn nhất là **{int(best_year['year'])}** với doanh thu **{best_year['revenue']:,.0f}**.

**Hàm ý / hành động**

Các tháng doanh thu cao không tự động đồng nghĩa với hiệu quả tốt nếu margin bị kéo xuống. Nhóm tháng có revenue cao nhưng margin thấp cần được kiểm tra thêm theo khuyến mãi, COGS và return. Với các tháng peak có tính lặp lại, tồn kho và logistics nên được chuẩn bị sớm hơn khoảng 4-8 tuần.
'''
display(Markdown(msg))

## Insight 2 - Product Portfolio Performance

Câu hỏi: nhóm sản phẩm nào vừa tạo doanh thu, vừa có margin tốt, vừa ít rủi ro trả hàng?


In [ ]:
cat_perf = df.groupby('category', as_index=False).agg(
    revenue=('net_revenue', 'sum'),
    gross_profit=('gross_profit', 'sum'),
    units=('quantity', 'sum'),
    orders=('order_id', 'nunique'),
    avg_margin=('gross_margin_rate', 'mean'),
    return_rate=('return_flag', 'mean'),
    avg_discount_rate=('discount_rate', 'mean')
)
cat_perf['profit_share'] = safe_divide(cat_perf['gross_profit'], cat_perf['gross_profit'].sum())
cat_perf['revenue_share'] = safe_divide(cat_perf['revenue'], cat_perf['revenue'].sum())
cat_perf = cat_perf.sort_values('gross_profit', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=cat_perf, y='category', x='gross_profit', ax=axes[0], palette='Set2')
axes[0].set_title('Gross Profit by Category')
axes[0].set_xlabel('Gross profit')
axes[0].set_ylabel('Category')

sns.scatterplot(data=cat_perf, x='revenue', y='avg_margin', size='return_rate', hue='category', sizes=(120, 900), ax=axes[1])
axes[1].set_title('Revenue vs Margin by Category\nBubble size = return rate')
axes[1].set_xlabel('Revenue')
axes[1].set_ylabel('Average margin rate')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIG_DIR / '02_category_portfolio.png', dpi=180, bbox_inches='tight')
plt.show()

display(cat_perf)

top_profit_cat = cat_perf.iloc[0]
top_margin_cat = cat_perf.loc[cat_perf['avg_margin'].idxmax()]
top_return_cat = cat_perf.loc[cat_perf['return_rate'].idxmax()]

msg = f'''
**Điểm chính**

- Category đóng góp gross profit cao nhất là **{top_profit_cat['category']}**, với gross profit **{top_profit_cat['gross_profit']:,.0f}**, chiếm **{top_profit_cat['profit_share']:.1%}** tổng gross profit.
- Category có margin trung bình cao nhất là **{top_margin_cat['category']}**, margin **{top_margin_cat['avg_margin']:.2%}**.
- Category có return rate cao nhất là **{top_return_cat['category']}**, return flag rate **{top_return_cat['return_rate']:.2%}**.

**Hàm ý / hành động**

Nếu chỉ nhìn revenue, một số nhóm sản phẩm có thể trông hấp dẫn hơn thực tế. Nhóm có gross profit cao và return rate thấp phù hợp để ưu tiên marketing. Ngược lại, nhóm doanh thu lớn nhưng return rate hoặc discount rate cao cần được rà soát về sizing, mô tả sản phẩm, chính sách khuyến mãi và fulfillment.
'''
display(Markdown(msg))

In [ ]:
seg_perf = df.groupby(['category', 'segment'], as_index=False).agg(
    revenue=('net_revenue', 'sum'),
    gross_profit=('gross_profit', 'sum'),
    units=('quantity', 'sum'),
    avg_margin=('gross_margin_rate', 'mean'),
    return_rate=('return_flag', 'mean')
)
seg_perf = seg_perf.sort_values('gross_profit', ascending=False).head(15)

plt.figure(figsize=(13, 7))
sns.barplot(data=seg_perf, y='segment', x='gross_profit', hue='category')
plt.title('Top Category-Segment Combinations by Gross Profit')
plt.xlabel('Gross profit')
plt.ylabel('Segment')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_top_segments_profit.png', dpi=180, bbox_inches='tight')
plt.show()

display(seg_perf)

## Insight 3 - Promotion Effectiveness

Câu hỏi: khuyến mãi đang tạo tăng trưởng thật hay chỉ đang mua doanh thu bằng cách hy sinh margin?


In [ ]:
promo_perf = df.groupby('has_promo', as_index=False).agg(
    rows=('order_id', 'count'),
    orders=('order_id', 'nunique'),
    revenue=('net_revenue', 'sum'),
    gross_profit=('gross_profit', 'sum'),
    avg_line_revenue=('net_revenue', 'mean'),
    avg_discount_rate=('discount_rate', 'mean'),
    avg_margin=('gross_margin_rate', 'mean'),
    return_rate=('return_flag', 'mean')
)
promo_perf['revenue_share'] = safe_divide(promo_perf['revenue'], promo_perf['revenue'].sum())

plot_df = promo_perf.melt(
    id_vars='has_promo',
    value_vars=['avg_line_revenue', 'avg_discount_rate', 'avg_margin', 'return_rate'],
    var_name='metric',
    value_name='value'
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=promo_perf, x='has_promo', y='revenue', ax=axes[0])
axes[0].set_title('Revenue: Promo vs Non-Promo Lines')
axes[0].set_xlabel('Has promotion')
axes[0].set_ylabel('Revenue')

sns.barplot(data=plot_df, x='metric', y='value', hue='has_promo', ax=axes[1])
axes[1].set_title('Promotion Trade-off Metrics')
axes[1].set_xlabel('Metric')
axes[1].set_ylabel('Value')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / '04_promotion_tradeoff.png', dpi=180, bbox_inches='tight')
plt.show()

display(promo_perf)

promo_yes = promo_perf[promo_perf['has_promo'] == True].iloc[0]
promo_no = promo_perf[promo_perf['has_promo'] == False].iloc[0]
margin_gap = promo_yes['avg_margin'] - promo_no['avg_margin']
return_gap = promo_yes['return_rate'] - promo_no['return_rate']

msg = f'''
**Điểm chính**

- Các dòng có promo chiếm **{promo_yes['rows'] / promo_perf['rows'].sum():.1%}** số dòng order item và **{promo_yes['revenue_share']:.1%}** doanh thu.
- Average discount rate của dòng có promo là **{promo_yes['avg_discount_rate']:.2%}**, so với **{promo_no['avg_discount_rate']:.2%}** ở dòng không promo.
- Chênh lệch margin giữa promo và non-promo là **{margin_gap:.2%}** điểm; return rate chênh lệch **{return_gap:.2%}** điểm.

**Hàm ý / hành động**

Promo nên được đánh giá bằng cả revenue, margin và return, không chỉ bằng doanh số. Các chiến dịch có điều kiện theo category, min order value hoặc nhóm khách hàng có khả năng kiểm soát margin tốt hơn discount đại trà.
'''
display(Markdown(msg))

In [ ]:
promo_detail = df.merge(
    promotions[['promo_id', 'promo_name', 'promo_type', 'discount_value', 'applicable_category', 'promo_channel', 'min_order_value']],
    on='promo_id', how='left'
)

promo_type_perf = promo_detail[promo_detail['has_promo']].groupby('promo_type', as_index=False).agg(
    rows=('order_id', 'count'),
    revenue=('net_revenue', 'sum'),
    gross_profit=('gross_profit', 'sum'),
    avg_discount_rate=('discount_rate', 'mean'),
    avg_margin=('gross_margin_rate', 'mean'),
    return_rate=('return_flag', 'mean')
).sort_values('gross_profit', ascending=False)

campaign_perf = promo_detail[promo_detail['has_promo']].groupby('promo_name', as_index=False).agg(
    revenue=('net_revenue', 'sum'),
    gross_profit=('gross_profit', 'sum'),
    avg_discount_rate=('discount_rate', 'mean'),
    avg_margin=('gross_margin_rate', 'mean'),
    return_rate=('return_flag', 'mean'),
    rows=('order_id', 'count')
).sort_values('gross_profit', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=promo_type_perf, x='promo_type', y='avg_margin', ax=axes[0])
axes[0].set_title('Average Margin by Promotion Type')
axes[0].set_xlabel('Promotion type')
axes[0].set_ylabel('Average margin')

sns.barplot(data=campaign_perf, y='promo_name', x='gross_profit', ax=axes[1])
axes[1].set_title('Top 10 Promotion Campaigns by Gross Profit')
axes[1].set_xlabel('Gross profit')
axes[1].set_ylabel('Promotion campaign')
plt.tight_layout()
plt.savefig(FIG_DIR / '05_promotion_campaigns.png', dpi=180, bbox_inches='tight')
plt.show()

display(promo_type_perf)
display(campaign_perf)

## Insight 4 - Returns, Delivery, and Customer Satisfaction

Câu hỏi: doanh nghiệp đang mất lợi nhuận ở đâu sau bán hàng, và nguyên nhân có liên quan tới sản phẩm hay logistics không?


In [ ]:
return_reason = returns.merge(products[['product_id', 'category', 'segment', 'size']], on='product_id', how='left')
reason_pivot = return_reason.pivot_table(
    index='category',
    columns='return_reason',
    values='return_id',
    aggfunc='count',
    fill_value=0
)

return_cat = df.groupby('category', as_index=False).agg(
    units_sold=('quantity', 'sum'),
    units_returned=('returned_qty', 'sum'),
    refund_amount=('refund_amount', 'sum'),
    revenue=('net_revenue', 'sum'),
    avg_delivery_days=('delivery_days', 'mean'),
    avg_rating=('avg_rating', 'mean')
)
return_cat['unit_return_rate'] = safe_divide(return_cat['units_returned'], return_cat['units_sold'])
return_cat['refund_rate'] = safe_divide(return_cat['refund_amount'], return_cat['revenue'])

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.heatmap(reason_pivot, annot=True, fmt='.0f', cmap='Reds', ax=axes[0])
axes[0].set_title('Return Reasons by Category')
axes[0].set_xlabel('Return reason')
axes[0].set_ylabel('Category')

sns.scatterplot(data=return_cat, x='avg_delivery_days', y='unit_return_rate', size='refund_rate', hue='category', sizes=(120, 900), ax=axes[1])
axes[1].set_title('Delivery Time vs Unit Return Rate\nBubble size = refund rate')
axes[1].set_xlabel('Average delivery days')
axes[1].set_ylabel('Unit return rate')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIG_DIR / '06_returns_delivery.png', dpi=180, bbox_inches='tight')
plt.show()

display(return_cat.sort_values('unit_return_rate', ascending=False))

top_reason = returns['return_reason'].value_counts().idxmax()
top_reason_count = returns['return_reason'].value_counts().max()
top_return_cat2 = return_cat.loc[return_cat['unit_return_rate'].idxmax()]
worst_refund_cat = return_cat.loc[return_cat['refund_rate'].idxmax()]

msg = f'''
**Điểm chính**

- Lý do trả hàng phổ biến nhất toàn bộ dữ liệu là **{top_reason}** với **{top_reason_count:,.0f}** bản ghi.
- Category có unit return rate cao nhất là **{top_return_cat2['category']}**, return rate **{top_return_cat2['unit_return_rate']:.2%}**.
- Category có refund rate cao nhất là **{worst_refund_cat['category']}**, refund/revenue **{worst_refund_cat['refund_rate']:.2%}**.

**Hàm ý / hành động**

Mỗi nhóm lý do trả hàng dẫn tới một hướng xử lý khác nhau: `wrong_size` gợi ý cải thiện size guide và ảnh fit thật; `late_delivery` gợi ý kiểm tra carrier hoặc vùng giao hàng; `defective` gợi ý rà soát supplier và quy trình QC.
'''
display(Markdown(msg))

In [ ]:
size_return = return_reason.groupby(['size', 'return_reason'], as_index=False).agg(
    returns=('return_id', 'count'),
    refund=('refund_amount', 'sum')
)
size_sold = df.groupby('size', as_index=False).agg(units_sold=('quantity', 'sum'))
size_return_total = return_reason.groupby('size', as_index=False).agg(units_returned=('return_quantity', 'sum'))
size_summary = size_sold.merge(size_return_total, on='size', how='left').fillna(0)
size_summary['unit_return_rate'] = safe_divide(size_summary['units_returned'], size_summary['units_sold'])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(data=size_summary.sort_values('unit_return_rate', ascending=False), x='size', y='unit_return_rate', ax=axes[0])
axes[0].set_title('Unit Return Rate by Size')
axes[0].set_xlabel('Size')
axes[0].set_ylabel('Unit return rate')

size_reason_pivot = size_return.pivot_table(index='size', columns='return_reason', values='returns', aggfunc='sum', fill_value=0)
sns.heatmap(size_reason_pivot, annot=True, fmt='.0f', cmap='Blues', ax=axes[1])
axes[1].set_title('Return Reasons by Size')
axes[1].set_xlabel('Return reason')
axes[1].set_ylabel('Size')
plt.tight_layout()
plt.savefig(FIG_DIR / '07_size_return_risk.png', dpi=180, bbox_inches='tight')
plt.show()

display(size_summary.sort_values('unit_return_rate', ascending=False))

## Insight 5 - Web Traffic Quality and Conversion Proxy

Câu hỏi: nguồn traffic nào tạo engagement tốt, và traffic có chuyển thành đơn hàng không?

Dữ liệu không có conversion ở cấp session, nên phần này dùng `daily_orders / sessions` làm proxy ở cấp ngày.


In [ ]:
daily_orders = orders.groupby('order_date', as_index=False).agg(
    orders=('order_id', 'nunique')
)

traffic_daily = web.groupby('date', as_index=False).agg(
    sessions=('sessions', 'sum'),
    unique_visitors=('unique_visitors', 'sum'),
    page_views=('page_views', 'sum'),
    bounce_rate=('bounce_rate', 'mean'),
    avg_session_duration_sec=('avg_session_duration_sec', 'mean')
)
traffic_orders = traffic_daily.merge(daily_orders, left_on='date', right_on='order_date', how='left')
traffic_orders['orders'] = traffic_orders['orders'].fillna(0)
traffic_orders['conversion_proxy'] = safe_divide(traffic_orders['orders'], traffic_orders['sessions'])
traffic_orders['month'] = traffic_orders['date'].dt.to_period('M').dt.to_timestamp()

source_perf = web.groupby('traffic_source', as_index=False).agg(
    days=('date', 'count'),
    sessions=('sessions', 'sum'),
    avg_bounce=('bounce_rate', 'mean'),
    avg_duration=('avg_session_duration_sec', 'mean'),
    page_views=('page_views', 'sum')
)
source_perf['views_per_session'] = safe_divide(source_perf['page_views'], source_perf['sessions'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=source_perf, x='avg_bounce', y='avg_duration', size='sessions', hue='traffic_source', sizes=(150, 900), ax=axes[0])
axes[0].set_title('Traffic Source Quality\nBubble size = sessions')
axes[0].set_xlabel('Average bounce rate')
axes[0].set_ylabel('Average session duration')
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')

sns.scatterplot(data=traffic_orders, x='bounce_rate', y='conversion_proxy', alpha=0.45, ax=axes[1])
axes[1].set_title('Daily Bounce Rate vs Conversion Proxy')
axes[1].set_xlabel('Bounce rate')
axes[1].set_ylabel('Orders / sessions')
plt.tight_layout()
plt.savefig(FIG_DIR / '08_traffic_quality.png', dpi=180, bbox_inches='tight')
plt.show()

display(source_perf.sort_values('avg_bounce'))

best_source = source_perf.loc[source_perf['avg_bounce'].idxmin()]
long_source = source_perf.loc[source_perf['avg_duration'].idxmax()]
corr = traffic_orders[['bounce_rate', 'conversion_proxy']].corr().iloc[0, 1]

msg = f'''
**Điểm chính**

- Traffic source có bounce rate thấp nhất là **{best_source['traffic_source']}**, average bounce **{best_source['avg_bounce']:.2%}**.
- Traffic source có session duration cao nhất là **{long_source['traffic_source']}**, average duration **{long_source['avg_duration']:,.1f} giây**.
- Tương quan giữa daily bounce rate và conversion proxy là **{corr:.3f}**.

**Hàm ý / hành động**

Nguồn traffic có session duration cao và bounce thấp là nhóm đáng xem xét tăng ngân sách. Với nguồn có nhiều sessions nhưng bounce cao, cần kiểm tra lại thông điệp quảng cáo, landing page và tệp target trước khi mở rộng.
'''
display(Markdown(msg))

## Insight 6 - Inventory Risk: Stockout vs Overstock

Câu hỏi: hàng bán tốt có bị thiếu không, và vốn có đang bị kẹt ở hàng chậm không?


In [ ]:
inv_summary = inventory.groupby('category', as_index=False).agg(
    stock_on_hand=('stock_on_hand', 'sum'),
    units_received=('units_received', 'sum'),
    units_sold=('units_sold', 'sum'),
    stockout_days=('stockout_days', 'sum'),
    avg_days_supply=('days_of_supply', 'mean'),
    fill_rate=('fill_rate', 'mean'),
    stockout_rate=('stockout_flag', 'mean'),
    overstock_rate=('overstock_flag', 'mean'),
    reorder_rate=('reorder_flag', 'mean'),
    sell_through_rate=('sell_through_rate', 'mean')
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=inv_summary, x='stockout_rate', y='overstock_rate', size='sell_through_rate', hue='category', sizes=(150, 900), ax=axes[0])
axes[0].set_title('Inventory Risk: Stockout vs Overstock\nBubble size = sell-through rate')
axes[0].set_xlabel('Stockout rate')
axes[0].set_ylabel('Overstock rate')
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')

sns.barplot(data=inv_summary.sort_values('fill_rate'), y='category', x='fill_rate', ax=axes[1])
axes[1].set_title('Average Fill Rate by Category')
axes[1].set_xlabel('Fill rate')
axes[1].set_ylabel('Category')
plt.tight_layout()
plt.savefig(FIG_DIR / '09_inventory_risk.png', dpi=180, bbox_inches='tight')
plt.show()

display(inv_summary.sort_values('stockout_rate', ascending=False))

stockout_cat = inv_summary.loc[inv_summary['stockout_rate'].idxmax()]
overstock_cat = inv_summary.loc[inv_summary['overstock_rate'].idxmax()]
low_fill_cat = inv_summary.loc[inv_summary['fill_rate'].idxmin()]

msg = f'''
**Điểm chính**

- Category có stockout rate cao nhất là **{stockout_cat['category']}**, stockout rate **{stockout_cat['stockout_rate']:.2%}**.
- Category có overstock rate cao nhất là **{overstock_cat['category']}**, overstock rate **{overstock_cat['overstock_rate']:.2%}**.
- Category có fill rate thấp nhất là **{low_fill_cat['category']}**, fill rate **{low_fill_cat['fill_rate']:.2%}**.

**Hàm ý / hành động**

Stockout cao ở nhóm bán tốt có thể làm mất doanh thu, trong khi overstock ở nhóm sell-through thấp làm kẹt vốn. Chính sách tồn kho nên tách theo rủi ro: tăng reorder threshold cho nhóm stockout cao + sell-through cao, và giảm nhập hoặc clearance cho nhóm overstock cao + sell-through thấp.
'''
display(Markdown(msg))

## Tổng hợp đề xuất

Bảng dưới đây chuyển các tín hiệu chính từ EDA thành hành động cụ thể cho revenue planning, product portfolio, promotion, returns, traffic và inventory.


In [ ]:
recommendations = pd.DataFrame([
    {'rubric_level': 'Predictive / Prescriptive', 'area': 'Revenue planning', 'signal': 'Doanh thu và margin biến động theo tháng', 'action': 'Dự báo nhu cầu theo mùa; chuẩn bị inventory và logistics trước tháng peak; kiểm tra các tháng revenue cao nhưng margin thấp.'},
    {'rubric_level': 'Diagnostic / Prescriptive', 'area': 'Product portfolio', 'signal': 'Category/segment có gross profit, margin và return rate khác nhau', 'action': 'Ưu tiên marketing cho nhóm margin cao + return thấp; rà soát nhóm revenue cao nhưng refund hoặc return cao.'},
    {'rubric_level': 'Diagnostic / Prescriptive', 'area': 'Promotion', 'signal': 'Promo có trade-off giữa revenue, discount, margin và return', 'action': 'Ưu tiên promo có điều kiện theo category, min order value và nhóm khách hàng thay vì discount đại trà.'},
    {'rubric_level': 'Diagnostic / Prescriptive', 'area': 'Returns and CX', 'signal': 'Return reason tập trung theo category/size', 'action': 'Nếu wrong_size: cải thiện size guide; nếu late_delivery: tối ưu carrier/vùng; nếu defective: kiểm soát QC/supplier.'},
    {'rubric_level': 'Diagnostic / Predictive', 'area': 'Traffic acquisition', 'signal': 'Traffic source khác nhau về bounce và session duration', 'action': 'Tăng ngân sách cho nguồn engagement tốt; kiểm tra nguồn sessions cao nhưng bounce cao trước khi mở rộng.'},
    {'rubric_level': 'Diagnostic / Prescriptive', 'area': 'Inventory', 'signal': 'Có category stockout cao và category overstock cao', 'action': 'Tăng reorder threshold cho nhóm stockout cao + sell-through cao; giảm nhập hoặc clearance cho nhóm overstock cao.'}
])
display(recommendations)

## Appendix - Baseline Forecasting

Mục này tạo `submission.csv` theo baseline dự báo: dùng `sales.csv` làm train, lấy ngày dự báo từ `sales_test.csv` nếu có hoặc `sample_submission.csv` nếu không có, rồi xuất đúng định dạng `Date,Revenue,COGS`.

Cách tính: seasonal profile theo `(month, day)` kết hợp tăng trưởng YoY trung bình hình học từ các năm đầy đủ 2013-2022.


In [ ]:
# Forecasting baseline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DATA_DIR = Path('dataset') if Path('dataset').exists() else Path('.')
TRAIN_FILE = BASE_DATA_DIR / 'sales.csv'
TEST_FILE = BASE_DATA_DIR / 'sales_test.csv'
SAMPLE_FILE = BASE_DATA_DIR / 'sample_submission.csv'
OUT_FILE = BASE_DATA_DIR / 'submission.csv'

if not TRAIN_FILE.exists():
    raise FileNotFoundError(f'Missing training file: {TRAIN_FILE}')

if TEST_FILE.exists():
    test_source = TEST_FILE
elif SAMPLE_FILE.exists():
    test_source = SAMPLE_FILE
else:
    raise FileNotFoundError('Need sales_test.csv or sample_submission.csv to get forecast dates.')

train_fc = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
test_fc = pd.read_csv(test_source, parse_dates=['Date'])[['Date']].copy()

for target in ['Revenue', 'COGS']:
    if target not in train_fc.columns:
        raise ValueError(f'Missing required target column in sales.csv: {target}')

train_fc['year'] = train_fc['Date'].dt.year
train_fc['month'] = train_fc['Date'].dt.month
train_fc['day'] = train_fc['Date'].dt.day

annual = train_fc.groupby('year')[['Revenue', 'COGS']].sum().sort_index()
full_years = annual.loc[2013:2022]
if len(full_years) < 2:
    raise ValueError('Need at least two full years between 2013 and 2022 to estimate YoY growth.')

yoy_rev = full_years['Revenue'].pct_change().dropna()
yoy_cogs = full_years['COGS'].pct_change().dropna()
growth_rev = (1 + yoy_rev).prod() ** (1 / len(yoy_rev))
growth_cogs = (1 + yoy_cogs).prod() ** (1 / len(yoy_cogs))

annual_means = train_fc.groupby('year')[['Revenue', 'COGS']].transform('mean')
train_fc['rev_norm'] = train_fc['Revenue'] / annual_means['Revenue']
train_fc['cogs_norm'] = train_fc['COGS'] / annual_means['COGS']

seasonal = (
    train_fc
    .groupby(['month', 'day'], as_index=False)[['rev_norm', 'cogs_norm']]
    .mean()
)

base_year = 2022
base_rev = annual.loc[base_year, 'Revenue'] / 365
base_cogs = annual.loc[base_year, 'COGS'] / 365

test_fc['year'] = test_fc['Date'].dt.year
test_fc['month'] = test_fc['Date'].dt.month
test_fc['day'] = test_fc['Date'].dt.day
test_fc['years_ahead'] = test_fc['year'] - base_year

test_fc = test_fc.merge(seasonal, on=['month', 'day'], how='left')
test_fc['rev_norm'] = test_fc['rev_norm'].fillna(1.0)
test_fc['cogs_norm'] = test_fc['cogs_norm'].fillna(1.0)

test_fc['Revenue'] = (base_rev * (growth_rev ** test_fc['years_ahead']) * test_fc['rev_norm']).round(2)
test_fc['COGS'] = (base_cogs * (growth_cogs ** test_fc['years_ahead']) * test_fc['cogs_norm']).round(2)

submission = test_fc[['Date', 'Revenue', 'COGS']].copy()
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission.to_csv(OUT_FILE, index=False)

print('Train shape:', train_fc.shape)
print('Train date range:', train_fc['Date'].min().date(), 'to', train_fc['Date'].max().date())
print('Forecast source:', test_source)
print('Forecast date range:', pd.to_datetime(submission['Date']).min().date(), 'to', pd.to_datetime(submission['Date']).max().date())
print(f'Geometric mean YoY Revenue growth: {(growth_rev - 1) * 100:.2f}%')
print(f'Geometric mean YoY COGS growth: {(growth_cogs - 1) * 100:.2f}%')
print(f'Saved {len(submission)} rows to {OUT_FILE}')
display(submission.head(10))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_fc['Date'], train_fc['Revenue'], lw=0.7, label='Historical Revenue')
ax.plot(pd.to_datetime(submission['Date']), submission['Revenue'], lw=0.9, label='Baseline Forecast Revenue')
ax.set_title('Baseline Revenue Forecast')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue')
ax.legend()
plt.tight_layout()
plt.show()